# Triangular Laplace--Beltrami equation on the sphere

Triangular-patch benchmark for the surface HPS method.


In [ ]:
from pathlib import Path
import sys
import time

import numpy as np

project = Path.cwd().resolve()
if project.name == 'notebooks':
    project = project.parent
sys.path.insert(0, str(project))

import pysurfacefun as psf

output_dir = project / 'notebook_outputs'
output_dir.mkdir(exist_ok=True)


## Geometry

A coarse spherical triangulation is refined once and projected to the unit sphere,

$$\phi(x,y,z)=x^2+y^2+z^2-1.$$


In [ ]:
mesh_file = project / 'notebook_data' / 'spher_104.mat'
mesh_refinement = 1

phi = lambda p: p[0]**2 + p[1]**2 + p[2]**2 - 1.0
dphi = lambda p: np.array([2.0*p[0], 2.0*p[1], 2.0*p[2]])

vertices, cells = psf.surface_mesh_arrays(mesh_file, vertex_name='xs', face_name='surfs')
_, refined_cells = psf.refine_surface_mesh(vertices, cells, nref=mesh_refinement, cell_type='tri')

print(f'coarse cells  : {cells.shape[0]}')
print(f'refined cells : {refined_cells.shape[0]}')


In [ ]:
def make_domain(n):
    return psf.LevelSetSurface(
        mesh_file,
        phi,
        dphi,
        n=n,
        nref=mesh_refinement,
        vertex_name='xs',
        face_name='surfs',
        orient_outward=True,
    )

dom = make_domain(n=15)
residual_phi = max(
    np.max(np.abs(x*x + y*y + z*z - 1.0))
    for x, y, z in zip(dom.x, dom.y, dom.z)
)
print(f'patches        : {dom.npatches}')
print(f'degree         : {dom.degree}')
print(f'area           : {psf.integral(psf.field(1.0, dom)):.12f}')
print(f'max |phi|      : {residual_phi:.3e}')


## Model problem

On the unit sphere,

$$\Delta_\Gamma Y_\ell^m = -\ell(\ell+1)Y_\ell^m.$$

The manufactured solution is $u=Y_{20}^{10}$.


In [ ]:
l, m = 20, 10

def exact_solution(dom):
    return psf.field(
        lambda x, y, z: psf.real_spherical_harmonic(l, m, x, y, z),
        dom,
    )

u_exact = exact_solution(dom)
rhs = -l * (l + 1) * u_exact


## Discrete operator

The reference differentiation matrices use the Proriol-Koornwinder-Dubiner basis.


In [ ]:
Du, Dv, K = psf.tri_strong_diffmat(dom.degree, dom.u, dom.v, basis='pkd')
print(f'K shape            : {K.shape}')
print(f'condition number K : {np.linalg.cond(K):.3e}')


## Solve

The mean is removed because the closed-surface Laplace-Beltrami operator has constants in the nullspace.


In [ ]:
problem = psf.SurfaceProblem(dom, variables='u', namespace={'rhs': rhs})
problem.add_equation('lap(u) = rhs')

t0 = time.perf_counter()
u_h = problem.build_solver(rankdef=True).solve().remove_mean()
elapsed = time.perf_counter() - t0

relerr = psf.norm(u_h - u_exact.remove_mean(), 'inf') / psf.norm(u_exact, 'inf')
print(f'relative L_inf error = {relerr:.3e}')
print(f'solve time           = {elapsed:.3f} s')

## p-refinement

The refined mesh is fixed and the polynomial degree is increased.


In [ ]:
n_values = [5,9,13,17,21]
rows = []

for n in n_values:
    dom_n = make_domain(n)
    exact = exact_solution(dom_n)
    rhs_n = -l * (l + 1) * exact

    problem_n = psf.SurfaceProblem(dom_n, variables='u', namespace={'rhs': rhs_n})
    problem_n.add_equation('lap(u) = rhs')

    t0 = time.perf_counter()
    sol = problem_n.build_solver(rankdef=True).solve().remove_mean()
    solve_time = time.perf_counter() - t0

    err = psf.norm(sol - exact.remove_mean(), 'inf') / psf.norm(exact, 'inf')
    rows.append((n, n - 1, dom_n.npatches, psf.integral(psf.field(1.0, dom_n)), err, solve_time))
    print(f'n = {n:2d}, p = {n-1:2d}, patches = {dom_n.npatches:3d}, rel error = {err:.3e}, time = {solve_time:.3f} s')

## Output


In [ ]:
try:
    psf.plot_tri_surface(u_h, title='Triangular Laplace--Beltrami solution')
except Exception as exc:
    print('surface plot skipped:', exc)

psf.write_tri_vtu(output_dir / 'tri_laplace_beltrami_sphere_refined_solution.vtu', u_h, point_name='u_h')


## Reference

G. Zavalani, *A High-Order Fast Direct Solver for Surface PDEs on Triangles*, arXiv:2604.03097, 2026.
